In a programming class, students are each assigned one of three different versions of a lab assignment. Students can only do code reviews for students doing a different version of the lab assignment. Students will organized into "lab partner" groups of three in order to do code reviews for each other. If the total number of students isn't a multiple of 3, there will be at least one group of 2.

Use the Z3 SMT Solver to assign lab versions (A, B or C) to the students listed below and organize them into their lab partner groups.

1. Aaliyah Jackson
2. Hiroshi Tanaka
3. Mateo Rodriguez
4. Chloe Dubois
5. Amara Okafor
6. Arjun Patel
7. Sofia Rossi
8. Dmitry Smirnov
9. Yasmine Al-Farsi
10. Ji-Woo Kim
11. Connor O'Connor
12. Fatima Bihi
13. Santiago Silva
14. Emily Chen
15. Malik Washington
16. Zoe Papadopoulos
17. Diego Gomez
18. Aisha Diop
19. Lucas Jensen
20. Mei-Ling Zhou



In [11]:
%pip install -q z3-solver

/Volumes/DataCard/Repos/CS212-Repos/CS212-CourseMaterials/docs/Examples/ConstraintSolving/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
from z3 import Solver, Int, Sum, If, Implies, sat

students = [
    "Aaliyah Jackson", "Hiroshi Tanaka", "Mateo Rodriguez", "Chloe Dubois",
    "Amara Okafor", "Arjun Patel", "Sofia Rossi", "Dmitry Smirnov",
    "Yasmine Al-Farsi", "Ji-Woo Kim", "Connor O'Connor", "Fatima Bihi",
    "Santiago Silva", "Emily Chen", "Malik Washington", "Zoe Papadopoulos",
    "Diego Gomez", "Aisha Diop", "Lucas Jensen", "Mei-Ling Zhou"
]
num_students = len(students)
print(num_students)

20


In [13]:
import math

MAX_GROUP_SIZE = 3
num_groups = math.ceil(num_students / MAX_GROUP_SIZE)

# Determine the breakdown of group sizes using the remainder operator:
remainder = num_students % MAX_GROUP_SIZE
if remainder == 0:
    num_groups_of_2 = 0
elif remainder == 2:
    num_groups_of_2 = 1
else:  # remainder == 1 (e.g., 4 students remaining: split into two groups of 2)
    num_groups_of_2 = 2

num_groups_of_3 = num_groups - num_groups_of_2

print(f"Total groups: {num_groups} ({num_groups_of_3} of size 3, {num_groups_of_2} of size 2)")

7


The `Int()` function (below) is used to create a symbolic integer placeholders called a ***decision variables*** to represent an unknown value, such as a student's group or lab version. Instead of holding fixed data, these variables allow you to write rules and relationships before the values are known. The solver then searches for concrete numbers to assign to each variable that satisfy all constraints.

In [14]:
# Lists of integer decision variables (one variable per student).
# These will contain the results produced by the solver.

# - group[i]: The lab partner group number (0 to num_groups - 1) assigned to student i:
group = [Int(f'group_{i}') for i in range(num_students)]
print(group)

# - version[i]: The lab version index (0=A, 1=B, 2=C) assigned to student i:
version = [Int(f'version_{i}') for i in range(num_students)]
print(version)

[group_0, group_1, group_2, group_3, group_4, group_5, group_6, group_7, group_8, group_9, group_10, group_11, group_12, group_13, group_14, group_15, group_16, group_17, group_18, group_19]
[version_0, version_1, version_2, version_3, version_4, version_5, version_6, version_7, version_8, version_9, version_10, version_11, version_12, version_13, version_14, version_15, version_16, version_17, version_18, version_19]


The `add()` method is used to add constraint expressions that use the ***decision variables*** declared previously. These constraint expressions are all put into one giant ***logical conjunction*** (they are all ANDed together).

In [15]:
s = Solver()

# Constraints on ranges
for i in range(num_students):
    s.add(group[i] >= 0, group[i] < num_groups)
    s.add(version[i] >= 0, version[i] < 3)

The `If(condition, then_value, else_value)` function is Z3's symbolic ternary operator. A standard Python `if` cannot be used here because the decision variables (`group[i]`) are unknown placeholders when the code executes. Instead, `If(group[i] == g, 1, 0)` acts as an indicator counter: it evaluates to `1` if student `i` is assigned to group `g`, and `0` otherwise.

When using an accumulator loop like `group_size += If(...)`, the `+=` operator is **overloaded** by Z3. It does not calculate a numeric sum; instead, it appends another term to the mathematical expression, expanding `group_size` into a longer formula with each iteration of the loop until all students are included.

In [16]:
# Group size constraints:
# The first num_groups_of_3 groups have 3 members, and any remaining groups have 2 members.
for g in range(num_groups):
    group_size = 0
    for i in range(num_students):
        group_size += If(group[i] == g, 1, 0)
    
    if g < num_groups_of_3:
        s.add(group_size == 3)
    else:
        s.add(group_size == 2)

The `Implies(condition, result)` function works like an "if-then" rule.

Here, it checks every pair of students and enforces that **if** two students are in the same group (`group[i] == group[j]`), **then** they must be assigned different lab versions (`version[i] != version[j]`).

In [ ]:
# Within each group, students must have different lab versions.
for i in range(num_students):
    for j in range(i + 1, num_students):
        # If student i and student j are in the same group, they must have different versions
        s.add(Implies(group[i] == group[j], version[i] != version[j]))

In [ ]:
if s.check() == sat:
    m = s.model()

    # Organize results
    groups_dict = {g: [] for g in range(num_groups)}
    versions_map = {0: 'A', 1: 'B', 2: 'C'}

    for i, name in enumerate(students):
        g_val = m[group[i]].as_long()
        v_val = m[version[i]].as_long()
        groups_dict[g_val].append((name, versions_map[v_val]))

    print("=== Lab Partner Groups and Assignments ===\n")
    for g in sorted(groups_dict.keys()):
        print(f"Group {g + 1} (Size {len(groups_dict[g])}):")
        for name, ver in groups_dict[g]:
            print(f"  - {name}: Version {ver}")
        print()
else:
    print("No valid grouping or version assignment found.")